# Shared Reliable-Cohort Explainability: Incongruent Mixed vs Incongruent Pure

This notebook interprets the IM vs IP task using a shared reliable-subject cohort. The goal is not to explain every trained model on every subject; the goal is to avoid interpreting noise while still comparing all 10 model families on the same subjects.

This contrast asks whether EEG separates two outcomes inside incongruent stimulation: mixed/piecemeal perception versus a resolved pure percept. Because both labels share physical incongruence, the interpretive bar is higher than for C vs IP: performance and attribution convergence both need to be convincing before the result should be discussed as perceptual-outcome evidence.

Main outputs:

- `selected_interpretability_subjects_im_vs_ip.csv`: the shared subjects used for all model maps.
- Spatial topomaps and temporal saliency curves for all 10 models.
- Peak-time model-minus-average topomaps for comparing model behavior.
- An experimental performance-weighted consensus diagnostic, clearly separated from thesis-grade evidence.


In [ ]:
# Unified Environment Setup (Local, Colab, Kaggle)
import os
import sys
import warnings
from pathlib import Path

IS_KAGGLE = False
IS_COLAB = (not IS_KAGGLE) and ("google.colab" in sys.modules or "COLAB_GPU" in os.environ)
IN_COLAB = IS_COLAB

if IS_KAGGLE:
    print("Detected Kaggle environment. Installing dependencies...")
    !pip install -q mne seaborn scikit-learn
elif IS_COLAB:
    print("Detected Google Colab environment. Installing dependencies...")
    !pip install -q mne seaborn scikit-learn
    from google.colab import drive
    drive.mount("/content/drive")

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

if IS_KAGGLE:
    dataset_root = None
    for root, dirs, files in os.walk("/kaggle/input"):
        if "br_analysis" in dirs:
            dataset_root = Path(root)
            break
    SRC_DIR = dataset_root if dataset_root else Path("/kaggle/input/nilsduranlaplaza/datasets/tfg-nils-2026-dataset")
elif IS_COLAB:
    SRC_DIR = Path("/content/drive/MyDrive/TFG_Nils_2026/03_Codi")
else:
    SRC_DIR = NOTEBOOK_DIR.parent.parent / "src"
    if not (SRC_DIR / "br_analysis" / "__init__.py").exists():
        SRC_DIR = NOTEBOOK_DIR.parent

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    from IPython import get_ipython
    ipy = get_ipython()
    if ipy is not None:
        ipy.run_line_magic("load_ext", "autoreload")
        ipy.run_line_magic("autoreload", "2")
except Exception:
    pass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown

from br_analysis import ProjectPaths, detect_project_paths, get_task_config
from br_analysis.explainability import (
    MODEL_COLORS,
    MODEL_LABELS,
    MODEL_ORDER,
    OOF_CACHE_VERSION,
    build_context,
    build_evidence_table,
    convergence_table,
    consensus_diagnostic,
    evidence_pyramid,
    load_oof_predictions,
    model_interpretation_table,
    plot_consensus_diagnostic,
    plot_peak_time_difference_topomaps,
    plot_shared_cohort_spatial_topomaps,
    plot_shared_cohort_temporal_saliency,
    selected_cohort_model_evidence,
    selected_subject_ids,
    plot_convergence,
    plot_evidence_overview,
    save_tables,
    synthesis_text,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
print(f"Using source directory: {SRC_DIR}")


In [ ]:
TASK_TAG = 'im_vs_ip'
TASK_TAG_DISPLAY = f"{TASK_TAG.split('_vs_')[0].upper()} vs {TASK_TAG.split('_vs_')[1].upper()}"
def resolve_project_paths(analysis_name):
    """Resolve the data/output project using local workspace candidates before external Drive defaults."""
    candidates = []
    if IS_COLAB:
        candidates.append(Path("/content/drive/MyDrive/TFG_Nils_2026"))
    for base in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents, Path.cwd(), *Path.cwd().parents]:
        candidates.extend([base / "TFG_Nils_2026", base.parent / "TFG_Nils_2026"])

    seen = set()
    for project_dir in candidates:
        project_dir = Path(project_dir)
        key = str(project_dir).lower()
        if key in seen:
            continue
        seen.add(key)
        analysis_candidates = [p for p in project_dir.glob("04_An*lisi") if p.is_dir()]
        analysis_root = analysis_candidates[0] if analysis_candidates else project_dir / "04_An?lisi"
        if analysis_root.exists():
            fixed_paths = ProjectPaths(
                project_dir=project_dir,
                data_dir=project_dir / "02_Data" / "01_Preprocessed_epochs",
                analysis_root=analysis_root,
                analysis_dir=analysis_root / analysis_name,
                cache_dir=analysis_root / analysis_name / "cache",
                tables_dir=analysis_root / analysis_name / "tables",
                figures_dir=analysis_root / analysis_name / "figures",
            )
            fixed_paths.ensure()
            print(f"Using data project directory: {project_dir}")
            return fixed_paths

    return detect_project_paths(analysis_name, in_colab=IN_COLAB, in_kaggle=IS_KAGGLE)

ANALYSIS_NAME = f"explainability_{TASK_TAG}"
paths = resolve_project_paths(ANALYSIS_NAME)
task = get_task_config(TASK_TAG, paths.analysis_root)
context = build_context(TASK_TAG, paths, task)

print(f"Project directory: {paths.project_dir}")
print(f"Analysis root: {paths.analysis_root}")
print(f"Notebook outputs: {paths.analysis_dir}")
print(f"Task: {context.display_name} ({context.class_0} vs {context.class_1})")
print(f"Expected cache version: {OOF_CACHE_VERSION}")


## 1. Task framing and expected neural hypotheses

This notebook is a sensor-level interpretability synthesis. It does not try to localize cortical sources; it asks whether the models rely on plausible EEG sensor patterns and time windows when they decode this task.

Two constraints guide the interpretation:

- **Performance first:** attribution maps are only scientifically useful when the corresponding model shows above-chance decoding evidence.
- **Shared subjects for comparison:** all 10 models are interpreted on the same reliable subject cohort for this task, so differences between maps are more likely to reflect model behavior rather than a different subject mix.

The expected patterns are deliberately modest: frontal/theta-region and posterior/alpha-region effects are treated as broad sensor-level hypotheses, not as proof of a specific neural generator.


In [ ]:
display(Markdown(
    f"**Expected interpretation for {TASK_TAG_DISPLAY}.**\n\n"
    f"{context.hypothesis}\n\n"
    f"{context.interpretation_default}"
))


## 2. Performance evidence and shared reliable-cohort selection

The first step is to decide which subjects are worth interpreting. This notebook ranks subjects using fresh out-of-fold predictions from every usable model, then selects one shared cohort for all model families.

Selection rule:

- Compute each subject's mean accuracy across usable models.
- Count how many models reach `accuracy >= 0.55` for that subject.
- Rank subjects by number of above-threshold models, then by mean accuracy.
- Use the top 5 reliable subjects when possible; fall back to top 3 if only three subjects are convincing.

The selected cohort is saved as `selected_interpretability_subjects_<task>.csv`. If the table is empty, the local runtime does not currently have the fresh OOF caches mounted/generated; the notebook is still structurally valid and will populate once those artifacts are available.


In [ ]:
evidence, diagnostics = build_evidence_table(paths, task, TASK_TAG)
selected_cohort = diagnostics['selected_cohort']
selected_subjects = selected_subject_ids(selected_cohort)
evidence = selected_cohort_model_evidence(evidence, diagnostics['subjects'], selected_subjects)
pyramid = evidence_pyramid(evidence)
convergence = convergence_table(
    evidence,
    paths,
    task,
    TASK_TAG,
    subjects=selected_subjects,
    weight_col='Selected cohort reliability weight',
)
consensus_table, consensus_vectors = consensus_diagnostic(
    evidence,
    paths,
    task,
    TASK_TAG,
    subjects=selected_subjects,
    df_subject=diagnostics['subjects'],
)
save_tables(
    paths,
    TASK_TAG,
    evidence,
    pyramid,
    convergence,
    selected_cohort=selected_cohort,
    consensus=consensus_table,
)

display_cols = [
    'Display',
    'Family',
    'Mean accuracy',
    'Selected cohort mean accuracy',
    'Selected cohort reliability weight',
    'N subjects',
    'Selected cohort N subjects',
    'Final holdout accuracy',
    'Chance/permutation status',
    'Attribution availability',
    'Attribution stability',
    'Confound risk level',
    'Thesis-use recommendation',
]

evidence_display = evidence[display_cols].copy()
for col in ['Mean accuracy', 'Selected cohort mean accuracy', 'Selected cohort reliability weight', 'Final holdout accuracy']:
    evidence_display[col] = evidence_display[col].map(lambda x: np.nan if pd.isna(x) else round(float(x), 4))

display(evidence_display)
display(Markdown(
    f"**Selected shared interpretability subjects:** {selected_subjects if selected_subjects else 'none available'}\n\n"
    f"Evidence and cohort tables saved to `{paths.tables_dir}`."
))



### Artifact freshness and selected subjects

This diagnostic block makes missing evidence visible instead of silently dropping models. A model is usable only when its OOF cache matches the expected cache version and grouped-development validation strategy.

The cohort table below is the critical one for the new workflow. The same selected subjects are reused for all 10 models in the later spatial, temporal, and consensus plots.


In [ ]:
cache_status = diagnostics['cache_status'].copy()
display(cache_status[['Model', 'OOF cache', 'OOF usable', 'OOF status']])

subjects = diagnostics['subjects']
display(Markdown(f"**Subjects with at least one fresh model artifact:** {len(subjects)}"))
if len(subjects):
    display(subjects.head())

display(Markdown('**Shared reliable-cohort ranking**'))
if selected_cohort.empty:
    display(Markdown('No subject-level accuracy artifacts are available for cohort selection in this runtime.'))
else:
    cohort_display = selected_cohort.copy()
    for col in ['mean_accuracy']:
        cohort_display[col] = cohort_display[col].map(lambda x: np.nan if pd.isna(x) else round(float(x), 4))
    display(cohort_display)



### Performance overview

This figure summarizes model accuracy before any interpretation. The dashed line marks nominal chance.

A model close to chance can still be plotted as a diagnostic, but it should not become thesis evidence just because its saliency map looks visually plausible. The selected-cohort reliability weight used later is `max(selected_cohort_mean_accuracy - 0.5, 0)`.


In [ ]:
fig = plot_evidence_overview(
    evidence,
    TASK_TAG,
    save_path=paths.figures_dir / f'explainability_performance_overview_{TASK_TAG}.png',
)
if fig is not None:
    plt.show()


### Selected-cohort ROC and precision-recall diagnostics

ROC and precision-recall curves are computed only from fresh OOF prediction arrays, and now only for the selected shared cohort. This keeps the discrimination curves aligned with the same subjects used in the interpretability maps.

If no curves appear, it means the fresh OOF prediction caches are missing or stale in the current runtime.


In [ ]:
from sklearn.metrics import auc, average_precision_score, precision_recall_curve, roc_curve

model_predictions = load_oof_predictions(paths, task, TASK_TAG, subjects=selected_subjects)
if not model_predictions:
    display(Markdown('No fresh OOF prediction arrays are available for the selected cohort, so ROC/PR diagnostics are skipped.'))
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=150)
    for model_name, preds in model_predictions.items():
        y_true = preds['y_true']
        y_prob = preds['y_prob']
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        axes[0].plot(
            fpr,
            tpr,
            lw=2,
            color=MODEL_COLORS.get(model_name, '#777777'),
            label=f"{MODEL_LABELS[model_name]} (AUC={auc(fpr, tpr):.3f})",
        )
        axes[1].plot(
            recall,
            precision,
            lw=2,
            color=MODEL_COLORS.get(model_name, '#777777'),
            label=f"{MODEL_LABELS[model_name]} (AP={average_precision_score(y_true, y_prob):.3f})",
        )
    axes[0].plot([0, 1], [0, 1], color='black', linestyle='--', linewidth=1)
    axes[0].set_title(f'Selected-cohort OOF ROC: {TASK_TAG_DISPLAY}')
    axes[0].set_xlabel('False positive rate')
    axes[0].set_ylabel('True positive rate')
    axes[0].legend(fontsize=8)
    axes[1].set_title('OOF precision-recall')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].legend(fontsize=8)
    sns.despine(fig=fig)
    fig.tight_layout()
    fig.savefig(paths.figures_dir / f'explainability_oof_roc_pr_{TASK_TAG}.png', dpi=150, bbox_inches='tight')
    plt.show()


## 3. Model-by-model interpretation

This table is the narrative checkpoint for the 10 models. It combines selected-cohort performance with attribution availability so each model has an explicit interpretation status.

A model can be useful in three different ways:

- **Evidence:** selected-cohort performance is meaningfully above chance and the attribution source is valid.
- **Diagnostic:** the model is plotted for comparison, but the reliability weight is weak or attribution is structural only.
- **Prediction-only:** fresh OOF predictions exist, but no saliency/IG/LRP/SHAP attribution has been exported for that model yet.

This avoids the earlier problem where missing DL saliency could look like a real zero-valued map.


In [ ]:
family_view = model_interpretation_table(evidence)
for col in ['Selected cohort accuracy', 'Reliability weight']:
    family_view[col] = family_view[col].map(lambda x: np.nan if pd.isna(x) else round(float(x), 4))
display(family_view)


## 4. Cross-model convergence and shared-cohort maps

This section asks whether the models converge on plausible sensor-level evidence. For linear/classical models, the maps reflect feature definitions or structural model priors. For deep models, the maps use cached saliency summaries when available.

The important methodological choice is that the selected subjects do not change from model to model. If a model performs poorly on the shared cohort, it remains visible with low or zero reliability weight rather than being replaced with easier model-specific subjects.


In [ ]:
conv_display = convergence.copy()
for col in ['Reliability weight', 'Frontal/theta-region alignment', 'Posterior/alpha-region alignment', 'Weighted convergence contribution']:
    conv_display[col] = conv_display[col].map(lambda x: round(float(x), 4))
display(conv_display)

fig = plot_convergence(
    convergence,
    TASK_TAG,
    save_path=paths.figures_dir / f'explainability_convergence_{TASK_TAG}.png',
)
plt.show()



### Shared-cohort spatial and temporal maps

These are the main interpretability figures.

- Spatial topomaps show each model's electrode importance on the shared cohort.
- Temporal saliency curves show when each model places the most importance during the analysis window.
- Low-weight models are still plotted, but their lines/maps should be read as diagnostics rather than evidence.

This is the safest comparison for linear vs non-linear models because every model is judged on the same subjects.


In [ ]:
fig = plot_shared_cohort_spatial_topomaps(
    evidence,
    paths,
    task,
    TASK_TAG,
    subjects=selected_subjects,
    save_path=paths.figures_dir / f'shared_cohort_spatial_topomaps_{TASK_TAG}.png',
)
plt.show()

fig = plot_shared_cohort_temporal_saliency(
    evidence,
    paths,
    task,
    TASK_TAG,
    subjects=selected_subjects,
    save_path=paths.figures_dir / f'shared_cohort_temporal_saliency_{TASK_TAG}.png',
)
plt.show()


### Peak-time model-minus-average maps

These plots summarize how each model differs from the cross-model average at its strongest temporal point.

The current saliency artifacts store spatial and temporal summaries separately, not full channel-by-time tensors. Because of that, this section uses each model's spatial pattern scaled at its temporal peak. It is a compact model-comparison diagnostic, not a claim that the exact full spatiotemporal attribution tensor has been recovered.


In [ ]:
fig = plot_peak_time_difference_topomaps(
    evidence,
    paths,
    task,
    TASK_TAG,
    subjects=selected_subjects,
    save_path=paths.figures_dir / f'peak_time_model_minus_average_topomaps_{TASK_TAG}.png',
)
if fig is not None:
    plt.show()


## 5. Experimental consensus diagnostic

This section builds the optional performance-weighted consensus map.

The weighting is intentionally simple: models receive weight only for selected-cohort accuracy above chance. The result is useful as an exploratory diagnostic of cross-model agreement, but it should not be described as a new global model unless it later receives its own validation.

The weighted consensus is compared with an unweighted consensus. If at least five selected subjects are available, the notebook also reports a bootstrap stability estimate for the selected-cohort mean accuracy.


In [ ]:
consensus_display = consensus_table.copy()
for col in [
    'Selected cohort reliability weight',
    'Temporal peak time (s)',
    'Weighted vs unweighted spatial cosine',
    'Weighted vs unweighted temporal cosine',
    'Bootstrap selected-subject mean-accuracy std',
]:
    consensus_display[col] = consensus_display[col].map(lambda x: np.nan if pd.isna(x) else round(float(x), 4))
display(consensus_display)

fig = plot_consensus_diagnostic(
    consensus_vectors,
    task,
    TASK_TAG,
    save_path=paths.figures_dir / f'experimental_consensus_diagnostic_{TASK_TAG}.png',
)
plt.show()


## 6. Confound diagnostics

This section lists the main reasons an attractive map might still be misleading.

The key risks are subject identity, temporal drift, class imbalance, cluster-selection circularity, stale caches, and overinterpretation of sensor topographies. The notebook keeps weak or stale models visible so the final interpretation can say clearly what is evidence, what is diagnostic, and what should be regenerated.


In [ ]:
confound_cols = [
    'Display',
    'OOF usable',
    'OOF status',
    'Final holdout status',
    'Chance/permutation status',
    'Confound risk level',
    'Thesis-use recommendation',
]
display(evidence[confound_cols])


## 7. Evidence pyramid

The evidence pyramid turns the previous diagnostics into a practical thesis-use ranking.

Use the top tier as the safest material for the thesis narrative. Treat weak or exploratory rows as supporting context only. Rows marked for discard/regeneration should not be used as scientific evidence until the underlying artifacts are refreshed.


In [ ]:
display(pyramid)


## 8. Final synthesis: what is probably truly important

The final paragraph is generated from the evidence table, selected-cohort reliability weights, and convergence diagnostics. It is meant to be a draft interpretation that can be edited after rerunning the notebook with the complete result artifacts.

If many rows are missing or stale, the correct conclusion is not that the models learned nothing; it is that this runtime does not currently contain the validation artifacts needed for interpretation.


In [ ]:
display(Markdown(synthesis_text(context, evidence, convergence)))
